In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import joblib

# Load Data
df = pd.read_csv('data/Feature_Engineering_Data.csv')

# Sort by date
df = df.sort_values(by=['store_id', 'department', 'date'])

# Handle NaNs from lag features
df = df.dropna()

# Sequential Time-Series split (80% train, 20% test)
split_idx = int(len(df) * 0.8)
train_set = df.iloc[:split_idx]
test_set = df.iloc[split_idx:]

X_train = train_set.drop(['weekly_sales', 'date', 'holiday_name'], axis=1, errors='ignore')
y_train = train_set['weekly_sales'].copy()
X_test = test_set.drop(['weekly_sales', 'date', 'holiday_name'], axis=1, errors='ignore')
y_test = test_set['weekly_sales'].copy()

print("Data loaded and split successfully!")

In [ ]:
# Column configuration
num_feats = ['store_size', 'temperature', 'fuel_price', 'markdown_1', 'markdown_2', 'markdown_3', 'markdown_4', 'markdown_5', 'cpi', 'unemployment', 'sales_lag_1', 'sales_lag_2', 'sales_lag_3', 'sales_lag_4', 'sales_lag_52', 'sales_roll_mean_4', 'sales_roll_std_4', 'sales_roll_mean_12', 'sales_roll_std_12', 'markdown_total', 'store_dept_week_avg']
cat_feats = ['store_id', 'department', 'is_holiday', 'store_type', 'region', 'season', 'week_of_year', 'month', 'year', 'quarter']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), 
    ('std_scaler', StandardScaler())               
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_feats),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats)
])

# Transform the Data
X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

print("Pipeline executed! Data shape:", X_train_prepared.shape)

## Model 1 - Random Forest

In [ ]:
print("Initializing and Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_prepared, y_train)

rf_preds = rf_model.predict(X_test_prepared)
print(f"RF RMSE: {np.sqrt(mean_squared_error(y_test, rf_preds)):.2f}")
print(f"RF MAE: {mean_absolute_error(y_test, rf_preds):.2f}")

## Model 2 - XGBoost

In [ ]:
print("Initializing and Training XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_prepared, y_train)

xgb_preds = xgb_model.predict(X_test_prepared)
print(f"XGB RMSE: {np.sqrt(mean_squared_error(y_test, xgb_preds)):.2f}")
print(f"XGB MAE: {mean_absolute_error(y_test, xgb_preds):.2f}")

## Model 3 - LSTM

In [ ]:
# Reshape 2D pipeline output to 3D for LSTM: [samples, time steps, features]
X_train_reshaped = np.expand_dims(X_train_prepared, axis=1)
X_test_reshaped = np.expand_dims(X_test_prepared, axis=1)

print("Initializing LSTM architecture...")
lstm_model = Sequential([
    LSTM(64, activation='relu', input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])),
    Dropout(0.2),
    Dense(1) # Output layer for regression
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("Training LSTM...")
history = lstm_model.fit(
    X_train_reshaped, y_train, 
    epochs=50, # Reduced to 50 for faster testing, push to 100 later if needed
    batch_size=32, 
    validation_split=0.1, 
    verbose=1
)

lstm_preds = lstm_model.predict(X_test_reshaped)
print(f"LSTM RMSE: {np.sqrt(mean_squared_error(y_test, lstm_preds)):.2f}")
print(f"LSTM MAE: {mean_absolute_error(y_test, lstm_preds):.2f}")

In [ ]:
# Example: Saving the XGBoost model and the pipeline
joblib.dump(xgb_model, 'xgb_model.pkl')
joblib.dump(full_pipeline, 'pipeline.pkl')
print("Model and pipeline saved for deployment!")